# VerseBridge Studio — Scripture in Motion

VersePulse brings Scripture into fitness and wearable experiences at the physiological moment encouragement matters most. This public notebook demonstrates the same deterministic biometric-moment engine used by the working web prototype. Live deployments retrieve licensed Scripture through YouVersion and generate source-grounded personalization through Gloo AI Studio; credentials are never embedded in this notebook.

In [ ]:
from pathlib import Path
import pandas as pd

biometric_files = list(Path('/kaggle/input').rglob('biometric movements.csv'))
mapping_files = list(Path('/kaggle/input').rglob('verse movement mapping.csv'))
if biometric_files and mapping_files:
    biometrics = pd.read_csv(biometric_files[0])
    mappings = pd.read_csv(mapping_files[0])
else:
    biometrics = pd.DataFrame([{'timestamp':'00:18:00','heart_rate':174,'hr_zone':5,'activity_type':'running','effort_pct':.91,'recovery_score':72,'stress_index':4.1,'session_minute':18}])
    mappings = pd.DataFrame([{'moment_type':'peak_effort','verse_reference':'ROM.8.37','verse_text_preview':'We are more than conquerors through him who loved us','translation':'NIV','theme_tag':'victory','delivery_format':'haptic_pulse + display','hr_zone_trigger':5,'effort_pct_trigger':.90,'activity_context':'all'}])
    print('Competition files are not mounted in this preview; running the documented official sample moment.')
print(f'{len(biometrics)} biometric moments · {len(mappings)} verse mappings')
biometrics.head()

## Moment detection

The wearable engine classifies peak effort, breakthrough walls, final reps, finishing strong, recovery, and steady state. Selection then prioritizes the detected moment, activity compatibility, and nearest effort trigger.

In [ ]:
def detect_moment(row):
    if row.hr_zone >= 5 or row.effort_pct >= .88:
        return 'peak_effort'
    if row.activity_type == 'weightlifting' and row.effort_pct >= .80:
        return 'final_rep'
    if row.hr_zone >= 4 and row.effort_pct >= .75:
        return 'breakthrough_wall'
    if row.session_minute >= 20 and row.effort_pct >= .70:
        return 'finishing_strong'
    if row.hr_zone <= 1 and row.effort_pct <= .25:
        return 'recovery'
    return 'steady_state'

def choose_delivery(row):
    moment = detect_moment(row)
    candidates = mappings[mappings.moment_type.eq(moment)].copy()
    if candidates.empty:
        candidates = mappings[mappings.moment_type.isin(['steady_state', 'recovery'])].copy()
    contexts = candidates.activity_context.fillna('all').str.split('/')
    candidates['context_match'] = contexts.map(lambda xs: row.activity_type in xs or 'all' in xs)
    candidates['distance'] = (candidates.effort_pct_trigger.fillna(0) - row.effort_pct).abs()
    winner = candidates.sort_values(['context_match', 'distance'], ascending=[False, True]).iloc[0]
    return pd.Series({
        'detected_moment': moment,
        'verse_reference': winner.verse_reference,
        'verse_preview': winner.verse_text_preview,
        'delivery_format': winner.delivery_format,
        'theme': winner.theme_tag,
    })

deliveries = biometrics.apply(choose_delivery, axis=1)
result = pd.concat([biometrics, deliveries], axis=1)
result[['timestamp','heart_rate','effort_pct','detected_moment','verse_reference','delivery_format']].head(12)

In [ ]:
sample = result.sort_values('effort_pct', ascending=False).iloc[0]
print('WEARABLE DELIVERY')
print(f"{sample.detected_moment.replace('_', ' ').title()} · {sample.heart_rate} BPM")
print(f"{sample.verse_reference} — {sample.verse_preview}")
print(f"{sample.delivery_format} · theme: {sample.theme}")

## Trust architecture

1. Biometric classification is deterministic and auditable.
2. Licensed Bible text is retrieved at runtime from YouVersion rather than copied into the repository.
3. Gloo AI receives only grounded passage text plus the creator brief.
4. Verified Scripture and generated commentary are visually separated.
5. Missing credentials activate clearly labeled demo mode rather than fabricated live output.

**Source and setup:** https://github.com/ILoveBuns/versebridge-studio